In [ ]:
import os
os.environ['OPENAI_API_KEY']="your_openai_api_key_here"  # Replace with your actual OpenAI API key

In [2]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 5.3 MB/s eta 0:00:00


In [82]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
#tool 1
@tool
def conversion_rate(b_c: str, t_c: str)->float:
  """This function fetches the currency conversion factor between a given base currency and a target currency"""
  url = f"https://v6.exchangerate-api.com/v6/<API_KEY>/pair/{b_c}/{t_c}" #Replace <API_KEY> with your actual API key from the ExchangeRate API service.
  response = requests.get(url)
  return response.json()

In [84]:
#tool 2
@tool
def convert(c_r:float, b_c:int)->float:
  """given a currency conversion rate this function calculates the target currency value from a given base currency value """
  return c_r*b_c

In [85]:
#tool binding:
llm = ChatOpenAI(model="gpt-6-luna")
llm_tools = llm.bind_tools([conversion_rate, convert])

In [86]:
msg = [HumanMessage('Based on the conversion rate, how much is the value of 10 dollars in rupees?')]

In [87]:
#tool calling:
a_msg = llm_tools.invoke(msg)
a_msg

AIMessage(content=[{'arguments': '{"b_c":"USD","t_c":"INR"}', 'call_id': 'call_sj1bHFEkD1Jax50dBgDugVIm', 'name': 'conversion_rate', 'type': 'function_call', 'id': 'fc_00b983f542a9a7e5006aca0e4aff8487d0ac861296ecf421a1', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_00b983f542a9a7e5006aca0e4a044c87d0b5edeb27ad54daf4', 'created_at': 1791626826.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_00b983f542a9a7e5006aca0e4a044c87d0b5edeb27ad54daf4', tool_calls=[{'name': 'conversion_rate', 'args': {'b_c': 'USD', 't_c': 'INR'}, 'id': 'call_sj1bHFEkD1Jax50dBgDugVIm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 26, 'total_tokens': 143, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [88]:
a_msg.tool_calls

[{'name': 'conversion_rate',
  'args': {'b_c': 'USD', 't_c': 'INR'},
  'id': 'call_sj1bHFEkD1Jax50dBgDugVIm',
  'type': 'tool_call'}]

In [89]:
tool_dict={
    conversion_rate.name : conversion_rate,
    convert.name : convert
}

In [ ]:
msg.append(a_msg)

while a_msg.tool_calls:
    t = a_msg.tool_calls[0]
    tool_name = t['name']
    tool = tool_dict[tool_name]
    t_msg = tool.invoke(t)
    msg.append(t_msg)
    a_msg = llm_tools.invoke(msg)
    msg.append(a_msg)
a_msg.text #final answer

'At the provided rate of 0.01032 USD per rupee, ₹57 is about **$0.59 USD**.'

In [ ]:
# The above implementation demonstrates a basic tool-calling workflow using the
# `conversion_rate` and `convert` tools.
#
# The `conversion_rate` tool retrieves the actual exchange rate from the
# ExchangeRate API.
#
# The `convert` tool uses the retrieved rate to perform the currency conversion.
#
# However, the model is responsible for providing all the arguments required by
# the `convert` tool, including the conversion rate (`c_r`).
#
# Therefore, the model must correctly pass the rate returned by the
# `conversion_rate` tool when making the subsequent `convert` tool call.
#
# The following implementation uses `InjectedToolArg` to give the application
# greater control over this process.
#
# It prevents the model from directly supplying the conversion rate. Instead,
# the application fetches the rate from the API and injects it into the
# `convert` tool during execution.
#
# This ensures that the conversion calculation uses the API-provided rate
# rather than depending on the model to pass it correctly.

second implementation:

In [123]:
import os
import requests

from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage

In [ ]:
@tool
def conversion_rate(b_c: str, t_c: str) -> float:
    """Get the actual conversion rate between two currencies."""

    url = (
        f"https://v6.exchangerate-api.com/v6/"
        f"<API_KEY>/pair/{b_c}/{t_c}" #Replace <API_KEY> with your actual API key from the ExchangeRate API service.
    )

    response = requests.get(url, timeout=15)
    response.raise_for_status()

    data = response.json()

    if data.get("result") != "success":
        raise ValueError(f"API error: {data}")

    return float(data["conversion_rate"])


@tool
def convert(
    amount: float,
    c_r: Annotated[float, InjectedToolArg]
) -> float:
    """Convert an amount using the supplied exchange rate."""

    return amount * c_r

In [131]:
llm = ChatOpenAI(model="gpt-6-luna")

llm_tools = llm.bind_tools([conversion_rate, convert])

tool_dict = {
    conversion_rate.name: conversion_rate,
    convert.name: convert
}

msg = [
    HumanMessage(
        content="Convert 10 USD to INR using the actual exchange rate."
    )
]

a_msg = llm_tools.invoke(msg)

In [132]:
msg.append(a_msg)

c_number = None

while a_msg.tool_calls:

    # Process every tool call returned in this AI response
    for t in a_msg.tool_calls:

        tool_name = t["name"]
        args = dict(t["args"])

        if tool_name == "conversion_rate":

            # Execute the exchange-rate tool
            c_number = tool_dict[tool_name].invoke(args)

            print("Conversion rate:", c_number)

            msg.append(
                ToolMessage(
                    content=str(c_number),
                    tool_call_id=t["id"]
                )
            )

        elif tool_name == "convert":

            # If the rate is missing, obtain it first
            if c_number is None:
                c_number = conversion_rate.invoke({
                    "b_c": "USD",
                    "t_c": "INR"
                })

                print("Conversion rate:", c_number)

            # Inject the actual rate into the convert tool
            args["c_r"] = c_number

            result = tool_dict[tool_name].invoke(args)

            print("Converted amount:", result)

            msg.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=t["id"]
                )
            )

    # Send tool results back to the LLM
    a_msg = llm_tools.invoke(msg)
    msg.append(a_msg)

print("Final answer:")
print(a_msg.content)

Conversion rate: 96.8685
Converted amount: 968.685
Final answer:
[{'type': 'text', 'text': '10 USD = 968.685 INR at the actual exchange rate provided.', 'annotations': [], 'id': 'msg_0baf0c01da82f68d006aca19c3aa6887d084c2e9e56a9491dc', 'phase': 'final_answer'}]
